# Tool Guardrails & Advanced Patterns

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to use `@tool_input_guardrail` and `@tool_output_guardrail` to validate individual tool calls, configure tool timeouts, use stop behaviors like `stop_on_first_tool` and `StopAtTools`, and handle tool failures gracefully.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Tool Input Guardrail

Validate tool arguments before the tool is executed. Here we block payments over $10,000.

In [ ]:
from agents import Agent, Runner, function_tool
from agents import tool_input_guardrail, ToolGuardrailFunctionOutput


@tool_input_guardrail
async def validate_amount(ctx, agent, tool_name, tool_input):
    """Block transactions over $10,000."""
    if tool_name == "process_payment":
        amount = tool_input.get("amount", 0)
        if amount > 10000:
            return ToolGuardrailFunctionOutput.reject_content(
                f"Amount ${amount} exceeds the $10,000 limit."
            )
    return ToolGuardrailFunctionOutput.allow()


@function_tool
def process_payment(amount: float, recipient: str) -> str:
    """Process a payment to a recipient."""
    return f"Payment of ${amount} sent to {recipient}"


payment_agent = Agent(
    name="Payment Agent",
    instructions="You process payments for users.",
    tools=[process_payment],
    tool_input_guardrails=[validate_amount],
)

result = await Runner.run(payment_agent, "Send $500 to Alice")
print(result.final_output)

## 4. Tool Input Guardrail — Blocked Request

The guardrail rejects the tool call when the amount exceeds the limit.

In [ ]:
result = await Runner.run(payment_agent, "Send $50,000 to Bob")
print(result.final_output)

## 5. Tool Output Guardrail

Inspect tool results after execution. Here we block outputs that contain sensitive credentials.

In [ ]:
from agents import tool_output_guardrail


@tool_output_guardrail
async def redact_secrets(ctx, agent, tool_name, tool_output):
    """Block tool outputs that contain API keys."""
    if "sk-" in str(tool_output) or "api_key" in str(tool_output).lower():
        return ToolGuardrailFunctionOutput.reject_content(
            "Tool output contained sensitive credentials and was blocked."
        )
    return ToolGuardrailFunctionOutput.allow()


@function_tool
def get_config(service: str) -> str:
    """Get configuration for a service."""
    configs = {
        "database": "host=db.example.com port=5432",
        "api": "endpoint=https://api.example.com",
    }
    return configs.get(service, "Service not found")


secure_agent = Agent(
    name="Secure Agent",
    instructions="You help users check service configurations.",
    tools=[get_config],
    tool_output_guardrails=[redact_secrets],
)

result = await Runner.run(secure_agent, "What's the database configuration?")
print(result.final_output)

## 6. Tool Timeouts

Set a timeout on tools to prevent long-running operations from blocking the agent.

In [ ]:
import asyncio


@function_tool(timeout=5, timeout_behavior="return_error")
async def slow_lookup(query: str) -> str:
    """Simulate a slow external API call."""
    await asyncio.sleep(2)
    return f"Result for {query}"


timeout_agent = Agent(
    name="Timeout Agent",
    instructions="You look up information for users.",
    tools=[slow_lookup],
)

result = await Runner.run(timeout_agent, "Look up information about Python.")
print(result.final_output)

## 7. Stop on First Tool

Use `tool_use_behavior="stop_on_first_tool"` to return the tool's raw output without further LLM processing.

In [ ]:
@function_tool
def lookup_user(user_id: str) -> str:
    """Look up a user by ID."""
    users = {"123": "Alice (alice@example.com)", "456": "Bob (bob@example.com)"}
    return users.get(user_id, "User not found")


lookup_agent = Agent(
    name="Lookup Agent",
    instructions="Look up the requested information.",
    tools=[lookup_user],
    tool_use_behavior="stop_on_first_tool",
)

result = await Runner.run(lookup_agent, "Find user #123")
print(result.final_output)

## 8. StopAtTools

Specify which tools should stop execution when called, while other tools continue normally.

In [ ]:
from agents import StopAtTools


@function_tool
def route_to_sales(reason: str) -> str:
    """Route the conversation to the sales team."""
    return f"Routed to sales: {reason}"


@function_tool
def route_to_support(reason: str) -> str:
    """Route the conversation to the support team."""
    return f"Routed to support: {reason}"


@function_tool
def general_lookup(query: str) -> str:
    """Look up general information."""
    return f"Info about: {query}"


router_agent = Agent(
    name="Router Agent",
    instructions="Route the user to the right department or look up general info.",
    tools=[route_to_sales, route_to_support, general_lookup],
    tool_use_behavior=StopAtTools(
        stop_at_tool_names=["route_to_sales", "route_to_support"]
    ),
)

result = await Runner.run(router_agent, "I want to buy the enterprise plan.")
print(result.final_output)

## 9. Failure Error Function

Customize the error message when a tool call fails.

In [ ]:
def custom_failure_handler(tool_name: str, error: Exception) -> str:
    return (
        f"The tool '{tool_name}' encountered an error: {error}. "
        f"Please try a different approach or ask the user for clarification."
    )


@function_tool
def risky_operation(data: str) -> str:
    """An operation that might fail."""
    if data == "bad":
        raise ValueError("Invalid data format")
    return f"Processed: {data}"


resilient_agent = Agent(
    name="Resilient Agent",
    instructions="You are a helpful assistant that handles errors gracefully.",
    tools=[risky_operation],
    failure_error_function=custom_failure_handler,
)

result = await Runner.run(resilient_agent, "Process this data: hello world")
print(result.final_output)

## Key Takeaways

- Use `@tool_input_guardrail` to validate tool arguments before execution
- Use `@tool_output_guardrail` to inspect and filter tool results after execution
- Call `ToolGuardrailFunctionOutput.reject_content()` to block and `.allow()` to pass through
- Set tool timeouts with `timeout` and `timeout_behavior` parameters
- Use `tool_use_behavior="stop_on_first_tool"` to return raw tool output
- Use `StopAtTools` to stop execution only on specific tools
- Customize tool failure messages with `failure_error_function`